## Adversarial Extraction Arena — Training (Colab)

Same notebook for **Extractor** and **Adversary** SFT (no second notebook).

**Scripts:** `training/sft_warmup.py` · `training/sft_adversary.py` · (optional) `training/grpo_trainer.py`

**Artifacts:** `checkpoints/sft_warmup/`, `checkpoints/sft_adversary/`, `trainer_log_history.json` in each · `plots/sft_loss.png`, `plots/sft_adversary_loss.png`

### Continuing later (same Colab or new runtime)
1. Run **Clone + pull** (next cell) so you get the latest GitHub code.
2. **Install** cell if packages are missing.
3. **Corpus** cell — if `corpus.json` is missing, run the generator cell once.
4. Run **Extractor** (cell 5) if you need `checkpoints/sft_warmup`; skip if you already have it in Drive or only want the adversary.
5. Run **Adversary** (cell 6) — only needs `data/corpus.json`, not the extractor weights.
6. **Plots** cell, then optional **HF push**.

### Notes
- **GPU**: `Runtime → Change runtime type → GPU`
- **Base model:** `unsloth/Qwen2.5-1.5B-Instruct`
- **Drive (recommended):** copy `checkpoints/` and `plots/` to Drive before disconnect — Colab discards local files when the session ends.


In [ ]:
# Clone once, then `git pull` every time you continue (fresh code for sft_adversary etc.)
import os
import subprocess

REPO = "openenv-adversarial-extraction-arena"
URL = "https://github.com/Hardikjha09/openenv-adversarial-extraction-arena.git"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", URL], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull"], check=False)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# Install dependencies (GPU runtime recommended)
!pip -q install -r requirements.txt

In [ ]:
# Verify corpus exists (skip generator if already present)
!python -c "import json; d=json.load(open('data/corpus.json','r',encoding='utf-8')); print('corpus_size=',len(d)); print('keys=',sorted(list(d[0].keys())))"

In [ ]:
# (Optional) Generate corpus if missing
# !PYTHONPATH=. python data/generator.py

In [ ]:
# Real training: Extractor SFT (writes checkpoints/sft_warmup + trainer_log_history.json)
!PYTHONPATH=. python training/sft_warmup.py \
  --model_name unsloth/Qwen2.5-1.5B-Instruct \
  --output_dir checkpoints/sft_warmup

In [ ]:
# Adversary SFT → checkpoints/sft_adversary/ (+ trainer_log_history.json)
# Needs data/corpus.json only (extractor checkpoint not required). Default slice [200:400] avoids
# overlapping the extractor's first-200 SFT docs; if corpus is shorter, the script falls back.
!PYTHONPATH=. python training/sft_adversary.py \
  --model_name unsloth/Qwen2.5-1.5B-Instruct \
  --output_dir checkpoints/sft_adversary \
  --start_idx 200 \
  --n_docs 200

In [ ]:
# (Optional) Real training: GRPO (writes checkpoints/grpo_extractor + trainer_log_history.json)
# !PYTHONPATH=. python training/grpo_trainer.py \
#   --model_name checkpoints/sft_warmup \
#   --output_dir checkpoints/grpo_extractor

In [ ]:
# Evidence: generate loss plots from trainer logs
!PYTHONPATH=. python plots/generate_training_plots.py

# List artifacts
!ls -la checkpoints/sft_warmup | head
!ls -la checkpoints/sft_adversary | head
!ls -la plots | head

In [ ]:
# (Optional) Quick two-model eval on holdout — needs BOTH checkpoints above. ~few min on T4 for 5 eps.
# !PYTHONPATH=. python evaluation/run_eval.py \
#   --model_path checkpoints/sft_warmup \
#   --adversary_model_path checkpoints/sft_adversary \
#   --num_episodes 5 \
#   --save_every 1

In [ ]:
# (Optional) Push trained checkpoints to HF model repos
# 1) Add HF_TOKEN in Colab secrets (or paste token carefully)
# 2) Then run:
# !pip -q install -U huggingface_hub
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(repo_id="HardikJha/extractor-aea", folder_path="checkpoints/sft_warmup", repo_type="model")
# api.upload_folder(repo_id="HardikJha/adversary-aea", folder_path="checkpoints/sft_adversary", repo_type="model")
# print("Extractor: https://huggingface.co/HardikJha/extractor-aea")
# print("Adversary: https://huggingface.co/HardikJha/adversary-aea")